# Mixed Precision Training

## Learning Objectives
1. Understand why FP16 overflows occur and how loss scaling mitigates them.
2. Demonstrate FP16 numerical limits vs FP32 using NumPy.
3. Implement Automatic Mixed Precision (AMP) with GradScaler in PyTorch.
4. Compare BF16 vs FP16 behaviour for training stability.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"FP16 max: {np.finfo(np.float16).max:.0f}")
print(f"FP32 max: {np.finfo(np.float32).max:.2e}")


## Level 1: FP16 Overflow Demonstration (NumPy)


In [ ]:
# Show the limits of FP16 and how simple operations overflow or underflow.

fp16_max = np.finfo(np.float16).max   # 65504
fp32_max = np.finfo(np.float32).max   # ~3.4e38

print(f"FP16 max value : {fp16_max}")
print(f"FP32 max value : {fp32_max:.2e}")

# Overflow: value exceeds FP16 max
a_fp16 = np.float16(60000.0)
b_fp16 = np.float16(10000.0)
overflow_result = a_fp16 + b_fp16
print(f"\n60000 + 10000 in FP16: {overflow_result}")
print(f"60000 + 10000 in FP32: {np.float32(60000.0) + np.float32(10000.0)}")

# Underflow in small gradients
small_gradient = np.float16(1e-7)
print(f"\n1e-7 in FP16: {small_gradient}")
print(f"1e-7 in FP32: {np.float32(1e-7)}")

# Loss scaling simulation
scale = 1024.0
scaled = np.float16(1e-7 * scale)
print(f"\n1e-7 * {scale} in FP16: {scaled}")
print(f"Unscaled back: {scaled / scale}")

# Numerical range comparison
formats = {'FP16': np.float16, 'FP32': np.float32, 'FP64': np.float64}
print("\nFormat   | Max value  | Min positive   | Mantissa bits")
print("-" * 56)
for name, dtype in formats.items():
    info = np.finfo(dtype)
    print(f"{name:<8} | {info.max:<10.2e} | {info.tiny:<14.2e} | {info.nmant}")


## Level 2: AMP with autocast and GradScaler (PyTorch + OOM Handling)


In [ ]:
class MLP(nn.Module):
    """Generic MLP for benchmark comparisons."""
    def __init__(self, in_dim=128, hidden=256, out_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)


def build_loader(n=2000, in_dim=128, n_classes=10, batch_size=64):
    X = torch.randn(n, in_dim)
    y = torch.randint(0, n_classes, (n,))
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)


loader = build_loader()
model_amp = MLP().to(device)
opt_amp = torch.optim.Adam(model_amp.parameters(), lr=1e-3)
scaler = GradScaler(device=device.type)
crit_amp = nn.CrossEntropyLoss()

amp_dtype = torch.bfloat16 if device.type == "cpu" else torch.float16

losses_amp = []
for epoch in range(30):
    model_amp.train()
    epoch_loss = 0.0
    for xb, yb in loader:
        opt_amp.zero_grad()
        try:
            with autocast(device_type=device.type, dtype=amp_dtype):
                logits = model_amp(xb.to(device))
                loss = crit_amp(logits, yb.to(device))
            scaler.scale(loss).backward()
            scaler.unscale_(opt_amp)
            nn.utils.clip_grad_norm_(model_amp.parameters(), 1.0)
            scaler.step(opt_amp)
            scaler.update()
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                print("OOM: reduce batch_size"); torch.cuda.empty_cache(); continue
            raise
        epoch_loss += loss.item()
    losses_amp.append(epoch_loss / len(loader))
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d} | AMP loss: {losses_amp[-1]:.4f} | "
              f"scale: {scaler.get_scale():.0f}")

print("AMP training complete. Final loss:", losses_amp[-1])


## Real-World Example 1: FP16 ResNet-Style Training Speedup


In [ ]:
import time

class TinyConvNet(nn.Module):
    """Small conv net that mimics ResNet structure for benchmarking."""
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Linear(64 * 4 * 4, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))


def benchmark_training(use_amp: bool, n_iters: int = 50) -> float:
    """Return average iteration time in ms."""
    model = TinyConvNet().to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    scaler_bench = GradScaler(device=device.type) if use_amp else None
    amp_dtype_b = torch.float16 if device.type == "cuda" else torch.bfloat16

    x = torch.randn(32, 3, 32, 32, device=device)
    y = torch.randint(0, 10, (32,), device=device)

    times = []
    for _ in range(n_iters):
        t0 = time.perf_counter()
        opt.zero_grad()
        if use_amp:
            with autocast(device_type=device.type, dtype=amp_dtype_b):
                logits = model(x)
                loss = crit(logits, y)
            scaler_bench.scale(loss).backward()
            scaler_bench.step(opt); scaler_bench.update()
        else:
            loss = crit(model(x), y)
            loss.backward(); opt.step()
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times[5:]))


t_fp32 = benchmark_training(use_amp=False)
t_amp = benchmark_training(use_amp=True)
print(f"FP32 avg iter time : {t_fp32:.2f} ms")
print(f"AMP  avg iter time : {t_amp:.2f} ms")
print(f"Speedup (AMP/FP32) : {t_fp32/t_amp:.2f}x")


## Real-World Example 2: Loss Scaling Underflow Demo


In [ ]:
def train_with_scaling(use_scaler: bool, n_epochs: int = 20):
    """Train with or without GradScaler and track gradient norms."""
    model_s = MLP(in_dim=64, hidden=128, out_dim=5).to(device)
    opt_s = torch.optim.Adam(model_s.parameters(), lr=1e-3)
    scaler_s = GradScaler(device=device.type) if use_scaler else None
    crit_s = nn.CrossEntropyLoss()
    loader_s = build_loader(n=500, in_dim=64, n_classes=5, batch_size=32)
    amp_dtype_s = torch.float16 if device.type == "cuda" else torch.bfloat16

    grad_norms, losses = [], []
    for epoch in range(n_epochs):
        for xb, yb in loader_s:
            opt_s.zero_grad()
            with autocast(device_type=device.type, dtype=amp_dtype_s):
                loss_s = crit_s(model_s(xb.to(device)), yb.to(device))
            if use_scaler:
                scaler_s.scale(loss_s).backward()
                scaler_s.unscale_(opt_s)
            else:
                loss_s.backward()

            total_norm = sum(
                p.grad.detach().float().norm().item() ** 2
                for p in model_s.parameters() if p.grad is not None
            ) ** 0.5
            grad_norms.append(total_norm)
            losses.append(loss_s.item())

            if use_scaler:
                scaler_s.step(opt_s); scaler_s.update()
            else:
                opt_s.step()
    return grad_norms, losses


norms_scaled, losses_scaled = train_with_scaling(use_scaler=True)
norms_unscaled, losses_unscaled = train_with_scaling(use_scaler=False)

print(f"With    GradScaler - median grad norm: {np.median(norms_scaled):.4f}, "
      f"zero-grad %: {100*np.mean(np.array(norms_scaled)<1e-8):.1f}%")
print(f"Without GradScaler - median grad norm: {np.median(norms_unscaled):.4f}, "
      f"zero-grad %: {100*np.mean(np.array(norms_unscaled)<1e-8):.1f}%")
print(f"Final loss (scaled): {losses_scaled[-1]:.4f}, "
      f"Final loss (unscaled): {losses_unscaled[-1]:.4f}")


## Real-World Example 3: BF16 vs FP16 Stability Comparison


In [ ]:
def train_format(dtype_name: str, n_epochs=25):
    """Train with a given dtype and return per-epoch losses."""
    dtype_map = {"fp32": None, "fp16": torch.float16, "bf16": torch.bfloat16}
    amp_dtype_f = dtype_map[dtype_name]
    model_f = MLP(in_dim=64, hidden=128, out_dim=5).to(device)
    opt_f = torch.optim.Adam(model_f.parameters(), lr=1e-3)
    scaler_f = GradScaler(device=device.type) if amp_dtype_f == torch.float16 else None
    crit_f = nn.CrossEntropyLoss()
    loader_f = build_loader(n=600, in_dim=64, n_classes=5, batch_size=32)
    epoch_losses = []
    for epoch in range(n_epochs):
        total = 0.0
        model_f.train()
        for xb, yb in loader_f:
            opt_f.zero_grad()
            if amp_dtype_f:
                with autocast(device_type=device.type, dtype=amp_dtype_f):
                    l = crit_f(model_f(xb.to(device)), yb.to(device))
                if scaler_f:
                    scaler_f.scale(l).backward(); scaler_f.step(opt_f); scaler_f.update()
                else:
                    l.backward(); opt_f.step()
            else:
                l = crit_f(model_f(xb.to(device)), yb.to(device))
                l.backward(); opt_f.step()
            total += l.item()
        epoch_losses.append(total / len(loader_f))
    return epoch_losses


results_fmt = {name: train_format(name) for name in ["fp32", "bf16", "fp16"]}

fig, ax = plt.subplots(figsize=(8, 4))
colors = {"fp32": "navy", "bf16": "steelblue", "fp16": "coral"}
for name, vals in results_fmt.items():
    ax.plot(vals, label=name.upper(), color=colors[name], linewidth=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("BF16 vs FP16 vs FP32 Training Loss")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/mixed_precision_formats.png", dpi=80)
plt.close()
print("Format comparison plot saved to /tmp/mixed_precision_formats.png")
for name, vals in results_fmt.items():
    print(f"{name.upper()} final loss: {vals[-1]:.4f}")
